In [1]:
!pip install implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=933263 sha256=4c5088de674ae327f7ccdf945a593a4b3106ef707853d7d84d7c520d0ce1faf3
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


In [ ]:
"""
NTO AI 2025/2026 Final — «Потеряшки» — Full Power CPU
2×ALS + KNN + CoOc + TF-IDF/SVD text + 2×CatBoost + rich features
"""
from __future__ import annotations
import gc, math, os, random, warnings
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
TOP_K = 20

@dataclass
class CFG:
    seed: int = 42
    data_root: Path = Path("/kaggle/input/datasets/artemnazemtsev/nto-ai")
    output_dir: Path = Path("/kaggle/working")
    n_folds: int = 3; fold_days: int = 30; hide_rate: float = 0.20
    read_w: float = 2.5; wish_w: float = 1.0; tau_full: float = 90.0; tau_recent: float = 21.0
    als1_f: int = 160; als1_reg: float = 0.04; als1_it: int = 15
    als2_f: int = 96; als2_reg: float = 0.06; als2_it: int = 12
    als1_topk: int = 180; als2_topk: int = 130
    knn_lat: int = 80; knn_sp: int = 60; cooc_k: int = 50
    txt_svd: int = 48; txt_topk: int = 40
    seed_n: int = 8; seed_h: int = 8; max_cands: int = 500
    top_au: int = 8; top_bk: int = 10; top_gn: int = 8
    top_ent: int = 25; top_demo: int = 25; top_glob: int = 40
    cb1_it: int = 2200; cb1_lr: float = 0.04
    cb2_it: int = 2200; cb2_lr: float = 0.035
    max_neg: int = 200; min_grp: int = 8; batch: int = 64

cfg = CFG()
if not cfg.data_root.exists():
    for p in [Path("/kaggle/input"), Path(".")]:
        if not p.exists(): continue
        for pp in p.rglob("interactions.csv"):
            cfg.data_root = pp.parent; break
cfg.output_dir.mkdir(parents=True, exist_ok=True)
random.seed(cfg.seed); np.random.seed(cfg.seed); os.environ["PYTHONHASHSEED"] = str(cfg.seed)
print("Data:", cfg.data_root)

# ══════════════════════════════════════════════════
# LOAD
# ══════════════════════════════════════════════════
print("Loading data...")
inter = pd.read_csv(cfg.data_root / "interactions.csv", low_memory=False)
targets = pd.read_csv(cfg.data_root / "targets.csv")
editions = pd.read_csv(cfg.data_root / "editions.csv", low_memory=False)
users_df = pd.read_csv(cfg.data_root / "users.csv")
authors_df = pd.read_csv(cfg.data_root / "authors.csv", low_memory=False)
book_genres = pd.read_csv(cfg.data_root / "book_genres.csv")

inter["event_ts"] = pd.to_datetime(inter["event_ts"])
inter = inter[inter["event_type"].isin([1, 2])].copy()
for c in ["user_id", "edition_id"]: inter[c] = inter[c].astype("int64")
inter["event_type"] = inter["event_type"].astype("int8")
inter["rating"] = pd.to_numeric(inter["rating"], errors="coerce").astype("float32")
inter = inter.sort_values("event_ts").reset_index(drop=True)

users_df["user_id"] = users_df["user_id"].astype("int64")
users_df["gender"] = pd.to_numeric(users_df["gender"], errors="coerce").fillna(-1).astype("int16")
users_df["age"] = pd.to_numeric(users_df["age"], errors="coerce").astype("float32")
users_df["age_bkt"] = pd.cut(users_df["age"].fillna(-1), bins=[-1,17,24,34,44,54,120],
    labels=[0,1,2,3,4,5]).astype("float").fillna(-1).astype("int16")
targets["user_id"] = targets["user_id"].astype("int64")

editions["edition_id"] = editions["edition_id"].astype("int64")
for c in ["book_id","author_id","language_id","publisher_id"]:
    editions[c] = pd.to_numeric(editions[c], errors="coerce").fillna(-1).astype("int64")
editions["pub_year"] = pd.to_numeric(editions["publication_year"], errors="coerce").astype("float32")
editions["age_restr"] = pd.to_numeric(editions["age_restriction"], errors="coerce").fillna(-1).astype("int16")
for c in ["title","description"]:
    if c not in editions.columns: editions[c] = ""
    editions[c] = editions[c].fillna("").astype(str)
editions["tlen"] = editions["title"].str.len().astype("int32")
editions["dlen"] = editions["description"].str.len().astype("int32")

book_genres["book_id"] = pd.to_numeric(book_genres["book_id"], errors="coerce").fillna(-1).astype("int64")
book_genres["genre_id"] = pd.to_numeric(book_genres["genre_id"], errors="coerce").fillna(-1).astype("int64")

# Edition meta
gc_df = book_genres.groupby("book_id")["genre_id"].nunique().reset_index(name="genre_cnt")
ed_meta = editions.merge(gc_df, on="book_id", how="left")
ed_meta["genre_cnt"] = ed_meta["genre_cnt"].fillna(0).astype("int16")
ed_meta["age_proxy"] = (2025 - ed_meta["pub_year"]).clip(-100, 300).fillna(0).astype("float32")
ed_meta["tdlen"] = (ed_meta["tlen"] + ed_meta["dlen"]).astype("int32")

# Edition-genre
ed_gn = editions[["edition_id","book_id"]].merge(book_genres, on="book_id", how="left")
ed_gn = ed_gn[ed_gn["genre_id"].notna() & (ed_gn["genre_id"]>=0)].copy()
ed_gn["genre_id"] = ed_gn["genre_id"].astype("int64")

# ══════════════════════════════════════════════════
# TEXT EMBEDDINGS (TF-IDF + SVD)
# ══════════════════════════════════════════════════
print("Building text embeddings...")
text_col = (editions["title"].fillna("") + " " + editions["description"].fillna("")).str.strip()
text_col = text_col.where(text_col.str.len() > 0, "empty")
tfidf = TfidfVectorizer(max_features=15000, sublinear_tf=True, dtype=np.float32,
    ngram_range=(1, 2), min_df=2, max_df=0.95)
tfidf_mat = tfidf.fit_transform(text_col)
svd = TruncatedSVD(n_components=cfg.txt_svd, random_state=cfg.seed)
txt_emb = svd.fit_transform(tfidf_mat).astype(np.float32)  # (n_editions, txt_svd)
txt_emb_n = normalize(txt_emb, norm="l2", axis=1)
eid_to_txtidx = {int(eid): i for i, eid in enumerate(editions["edition_id"].values)}
txt_nn = NearestNeighbors(metric="cosine", algorithm="auto", n_jobs=-1).fit(txt_emb_n)
del tfidf_mat, tfidf; gc.collect()
print(f"  Text embeddings: {txt_emb.shape}, explained var: {svd.explained_variance_ratio_.sum():.3f}")

# Drop raw text
editions.drop(columns=["title","description","publication_year","age_restriction"], inplace=True, errors="ignore")
gc.collect()

for k, v in {"inter": inter, "targets": targets, "editions": editions}.items():
    print(f"  {k}: {v.shape}")

# ══════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════
def ndcg(pred, rel, k=TOP_K):
    if not rel: return 0.0
    dcg = sum(1.0/math.log2(i+2) for i,x in enumerate(pred[:k]) if x in rel)
    idcg = sum(1.0/math.log2(i+2) for i in range(min(len(rel), k)))
    return dcg/idcg if idcg > 0 else 0.0

def ev_scored(sdf, hid, scol, k=TOP_K):
    rel = hid.groupby("user_id")["edition_id"].agg(lambda x: set(int(v) for v in x)).to_dict()
    tot, cnt = 0.0, 0
    for uid, g in sdf.sort_values(["user_id",scol], ascending=[True,False]).groupby("user_id"):
        r = rel.get(int(uid), set())
        if not r: continue
        tot += ndcg(g["edition_id"].head(k).astype(int).tolist(), r, k); cnt += 1
    return tot / max(cnt, 1)

def r2s(s, shift=1.0): return 1.0 / (s.astype(float) + shift)

# ══════════════════════════════════════════════════
# WEIGHTS & PAIRS
# ══════════════════════════════════════════════════
def add_w(df, ref):
    df = df.copy()
    age = ((ref - df["event_ts"]).dt.total_seconds() / 86400.0).clip(lower=0)
    ew = np.where(df["event_type"]==2, cfg.read_w, cfg.wish_w).astype("float32")
    rat = np.clip(np.where(df["rating"].notna(), 1+0.08*(df["rating"].fillna(0).values-3), 1), 0.65, 1.35).astype("float32")
    df["is_rd"] = (df["event_type"]==2).astype("int8")
    df["is_wl"] = (df["event_type"]==1).astype("int8")
    df["wf"] = ew * (0.35 + 0.65*np.exp(-age/cfg.tau_full)) * rat
    df["wr"] = ew * np.exp(-age/cfg.tau_recent) * rat
    return df

def mk_pairs(obs, ref):
    w = add_w(obs, ref)
    p = w.groupby(["user_id","edition_id"], as_index=False).agg(
        n=("event_ts","size"), rds=("is_rd","sum"), wls=("is_wl","sum"),
        wf=("wf","sum"), wr=("wr","sum"), lt=("event_ts","max"),
        ft=("event_ts","min"), mx_r=("rating","max"), mn_r=("rating","mean"))
    p["ld"] = ((ref - p["lt"]).dt.total_seconds()/86400).astype("float32")
    p["fd"] = ((ref - p["ft"]).dt.total_seconds()/86400).astype("float32")
    del w; gc.collect()
    return p

# ══════════════════════════════════════════════════
# ALS
# ══════════════════════════════════════════════════
try:
    import implicit; HAS_IMP = True
except: HAS_IMP = False
print("implicit:", HAS_IMP)

def mk_mat(pdf, vcol):
    uids = np.sort(pdf["user_id"].unique()); iids = np.sort(pdf["edition_id"].unique())
    u2i = {int(u): i for i, u in enumerate(uids)}; i2i = {int(x): i for i, x in enumerate(iids)}
    r = pdf["user_id"].map(u2i).values; c = pdf["edition_id"].map(i2i).values
    v = pdf[vcol].values.astype(np.float32)
    m = sparse.csr_matrix((v,(r,c)), shape=(len(uids), len(iids))); m.sum_duplicates()
    return m, u2i, iids

def nrm(x):
    d = np.linalg.norm(x, axis=1, keepdims=True); return x / np.where(d<1e-12, 1, d)

class ALS:
    def __init__(s, f, reg, it, seed):
        s.f,s.reg,s.it,s.seed = f,reg,it,seed
        s.u2i={}; s.i2i={}; s.idx=None; s.seen=None
        s.uf=None; s.itf=None; s.itf_n=None
    def fit(s, pdf, vcol):
        mat, s.u2i, s.idx = mk_mat(pdf, vcol)
        s.i2i = {int(x): i for i, x in enumerate(s.idx)}
        s.seen = mat.tocsr(); nu, ni = mat.shape
        if HAS_IMP:
            try:
                m = implicit.als.AlternatingLeastSquares(
                    factors=s.f, regularization=s.reg, iterations=s.it,
                    random_state=s.seed, use_gpu=False)
                try: m.fit(mat, show_progress=True)
                except: m.fit(mat.T.tocsr(), show_progress=True)
                uf, itf = m.user_factors, m.item_factors
                if hasattr(uf,"to_numpy"): uf=uf.to_numpy()
                if hasattr(itf,"to_numpy"): itf=itf.to_numpy()
                uf, itf = np.array(uf, dtype=np.float32), np.array(itf, dtype=np.float32)
                if uf.shape[0]==ni and itf.shape[0]==nu: uf, itf = itf, uf
                s.uf, s.itf = uf, itf; del m; gc.collect()
                print(f"  ALS: u={s.uf.shape} i={s.itf.shape}")
            except Exception as e:
                print(f"  ALS fallback: {e}"); s._fb(mat)
        else: s._fb(mat)
        s.itf_n = nrm(s.itf); del mat; gc.collect()
        return s
    def _fb(s, mat):
        f=min(s.f,48); it=min(s.it,6); rng=np.random.default_rng(s.seed)
        s.uf=(0.01*rng.standard_normal((mat.shape[0],f))).astype(np.float32)
        s.itf=(0.01*rng.standard_normal((mat.shape[1],f))).astype(np.float32)
        eye=np.eye(f,dtype=np.float32); Ciu=mat.T.tocsr()
        for _ in range(it):
            s.uf=s._sol(mat,s.itf,eye); s.itf=s._sol(Ciu,s.uf,eye)
    def _sol(s, C, Y, eye):
        X=np.zeros((C.shape[0],Y.shape[1]),dtype=np.float32); YtY=Y.T@Y
        for u in range(C.shape[0]):
            st,en=C.indptr[u],C.indptr[u+1]; idx,conf=C.indices[st:en],C.data[st:en]
            if len(idx)==0: continue
            A=YtY+s.reg*eye; b=np.zeros(Y.shape[1],dtype=np.float32)
            for i,c in zip(idx,conf): y=Y[i]; A+=(c-1)*np.outer(y,y); b+=c*y
            X[u]=np.linalg.solve(A,b)
        return X
    def score_pairs(s, uids, iids):
        sc=np.zeros(len(uids),dtype=np.float32); cos=np.zeros(len(uids),dtype=np.float32)
        if s.uf is None: return sc, cos
        mask=np.array([int(u) in s.u2i and int(i) in s.i2i for u,i in zip(uids,iids)])
        if mask.any():
            ui=np.array([s.u2i[int(u)] for u in uids[mask]]); ii=np.array([s.i2i[int(i)] for i in iids[mask]])
            uv,iv=s.uf[ui],s.itf[ii]; sc[mask]=np.sum(uv*iv,axis=1)
            un=np.linalg.norm(uv,axis=1); ine=np.linalg.norm(iv,axis=1)
            cos[mask]=sc[mask]/np.maximum(un*ine,1e-12)
        return sc.astype(np.float32), cos.astype(np.float32)
    def recommend(s, uids, n):
        rows=[]; valid=[int(u) for u in uids if int(u) in s.u2i]
        if not valid or s.uf is None: return pd.DataFrame(columns=["user_id","edition_id","rank","score"])
        ni=s.itf.shape[0]
        for st in range(0, len(valid), cfg.batch):
            batch=valid[st:st+cfg.batch]; bi=np.array([s.u2i[u] for u in batch])
            scores=s.uf[bi]@s.itf.T
            for ri in range(len(batch)):
                si=s.seen[bi[ri]].indices; si=si[si<ni]; scores[ri,si]=-1e18
            k=min(n,ni)
            if k<=0: continue
            top=np.argpartition(-scores, min(k-1,ni-1), axis=1)[:,:k]
            for ri,uid in enumerate(batch):
                ix=top[ri]; sc=scores[ri,ix]; o=np.argsort(-sc); ix,sc=ix[o],sc[o]
                rows.append(pd.DataFrame({"user_id":uid,"edition_id":s.idx[ix].astype(np.int64),
                    "rank":np.arange(1,len(ix)+1,dtype=np.int32),"score":sc.astype(np.float32)}))
            del scores
        return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["user_id","edition_id","rank","score"])

# ══════════════════════════════════════════════════
# KNN + COOC + TEXT RETRIEVAL
# ══════════════════════════════════════════════════
def fit_knn(pdf, als):
    lat_nn=None
    if als.itf_n is not None and len(als.itf_n)>1:
        lat_nn=NearestNeighbors(metric="cosine",algorithm="auto",n_jobs=-1).fit(als.itf_n)
    mat,_,idx=mk_mat(pdf,"wr"); iu=normalize(mat.T.tocsr(),norm="l2",axis=1)
    sp_nn=NearestNeighbors(metric="cosine",algorithm="brute",n_jobs=-1).fit(iu) if iu.shape[0]>1 else None
    sp_i2i={int(x):i for i,x in enumerate(idx)}
    del mat; gc.collect()
    return {"lat_nn":lat_nn,"sp_nn":sp_nn,"sp_iu":iu,"sp_idx":idx,"sp_i2i":sp_i2i}

def q_nbrs(als, seeds, nn, topk):
    if nn is None or als.itf_n is None: return pd.DataFrame(columns=["sid","edition_id","nr","sim"])
    valid=[int(x) for x in np.unique(seeds) if int(x) in als.i2i]
    if not valid: return pd.DataFrame(columns=["sid","edition_id","nr","sim"])
    idx=np.array([als.i2i[x] for x in valid]); k=min(topk+1,als.itf_n.shape[0])
    d,ind=nn.kneighbors(als.itf_n[idx],n_neighbors=k); rows=[]
    for si,dr,ir in zip(valid,d,ind):
        sim=(1.0-dr).astype(np.float32); items=als.idx[ir].astype(np.int64)
        df=pd.DataFrame({"sid":int(si),"edition_id":items,"sim":sim})
        df=df[df["edition_id"]!=int(si)].head(topk)
        df["nr"]=np.arange(1,len(df)+1,dtype=np.int32); rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["sid","edition_id","nr","sim"])

def q_sp_nbrs(seeds, ki, topk):
    nn=ki["sp_nn"]
    if nn is None: return pd.DataFrame(columns=["sid","edition_id","nr","sim"])
    valid=[int(x) for x in np.unique(seeds) if int(x) in ki["sp_i2i"]]
    if not valid: return pd.DataFrame(columns=["sid","edition_id","nr","sim"])
    idx=np.array([ki["sp_i2i"][x] for x in valid]); k=min(topk+1,ki["sp_iu"].shape[0])
    d,ind=nn.kneighbors(ki["sp_iu"][idx],n_neighbors=k); rows=[]
    for si,dr,ir in zip(valid,d,ind):
        sim=(1.0-dr).astype(np.float32); items=ki["sp_idx"][ir].astype(np.int64)
        df=pd.DataFrame({"sid":int(si),"edition_id":items,"sim":sim})
        df=df[df["edition_id"]!=int(si)].head(topk)
        df["nr"]=np.arange(1,len(df)+1,dtype=np.int32); rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["sid","edition_id","nr","sim"])

def txt_cands(seeds_df, user_ids, topk):
    """Content-based retrieval via text embeddings."""
    seed_sub = seeds_df[seeds_df["user_id"].isin(user_ids)]
    rows = []
    for uid in seed_sub["user_id"].unique():
        us = seed_sub[seed_sub["user_id"] == uid]
        txidx, txw = [], []
        for _, row in us.iterrows():
            eid = int(row["edition_id"])
            if eid in eid_to_txtidx:
                txidx.append(eid_to_txtidx[eid]); txw.append(float(row["sw"]))
        if not txidx: continue
        # Weighted average of seed embeddings
        w = np.array(txw, dtype=np.float32); w /= w.sum()
        qvec = (txt_emb_n[txidx] * w[:, None]).sum(axis=0, keepdims=True)
        qvec = normalize(qvec, norm="l2")
        d, ind = txt_nn.kneighbors(qvec, n_neighbors=min(topk + len(txidx) + 1, txt_emb_n.shape[0]))
        sim = (1.0 - d[0]).astype(np.float32)
        eids = editions["edition_id"].values[ind[0]].astype(np.int64)
        seed_set = set(us["edition_id"].astype(int).tolist())
        mask = np.array([int(e) not in seed_set for e in eids])
        eids, sim = eids[mask][:topk], sim[mask][:topk]
        if len(eids) == 0: continue
        df = pd.DataFrame({"edition_id": eids, "txt_sim": sim})
        df["user_id"] = int(uid); df["txt_rank"] = np.arange(1, len(df)+1, dtype=np.int32)
        rows.append(df[["user_id","edition_id","txt_sim","txt_rank"]])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["user_id","edition_id","txt_sim","txt_rank"])

def build_cooc(pdf, seeds_df, user_ids):
    """Co-occurrence via sparse matmul — fully sparse, no todense."""
    uids_a=np.sort(pdf["user_id"].unique()); iids_a=np.sort(pdf["edition_id"].unique())
    u2i={int(u):i for i,u in enumerate(uids_a)}; i2i={int(x):i for i,x in enumerate(iids_a)}
    r=pdf["user_id"].map(u2i).values; c=pdf["edition_id"].map(i2i).values
    ui=sparse.csr_matrix((np.ones(len(r),dtype=np.float32),(r,c)),shape=(len(uids_a),len(iids_a)))
    seed_sub=seeds_df[seeds_df["user_id"].isin(user_ids)]; rows=[]
    for uid in seed_sub["user_id"].unique():
        if int(uid) not in u2i: continue
        us=seed_sub[seed_sub["user_id"]==uid]; si,sw=[],[]
        for _,row in us.iterrows():
            eid=int(row["edition_id"])
            if eid in i2i: si.append(i2i[eid]); sw.append(float(row["sw"]))
        if not si: continue
        sv=sparse.csr_matrix((sw,([0]*len(si),si)),shape=(1,len(iids_a)))
        user_sc=ui @ sv.T  # sparse (n_users, 1)
        user_sc[u2i[int(uid)], 0] = 0
        item_sc=ui.T @ user_sc  # sparse (n_items, 1)
        # Extract scores from sparse column vector
        item_sc_coo = item_sc.tocoo()
        nz_row = item_sc_coo.row
        nz_val = item_sc_coo.data.astype(np.float32)
        if len(nz_row) == 0: continue
        # Remove seeds and seen
        si_set = set(si)
        seen_set = set(ui[u2i[int(uid)]].indices.tolist())
        keep = np.array([r not in si_set and r not in seen_set for r in nz_row])
        nz_row, nz_val = nz_row[keep], nz_val[keep]
        if len(nz_row) == 0: continue
        tk=cfg.cooc_k*3
        if len(nz_row) > tk:
            top=np.argpartition(-nz_val, tk)[:tk]; nz_row,nz_val=nz_row[top],nz_val[top]
        order=np.argsort(-nz_val); nz_row,nz_val=nz_row[order],nz_val[order]
        df=pd.DataFrame({"edition_id":iids_a[nz_row].astype(np.int64),"cooc_sc":nz_val})
        df["user_id"]=int(uid); df["cooc_rk"]=np.arange(1,len(df)+1,dtype=np.int32)
        rows.append(df[["user_id","edition_id","cooc_sc","cooc_rk"]])
    del ui; gc.collect()
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["user_id","edition_id","cooc_sc","cooc_rk"])

# ══════════════════════════════════════════════════
# CONTEXT BUILDER
# ══════════════════════════════════════════════════
def build_ctx(obs, ref):
    ow = add_w(obs, ref); pair = mk_pairs(obs, ref)
    # -- User features --
    uf = pair.groupby("user_id", as_index=False).agg(
        u_items=("edition_id","nunique"), u_ev=("n","sum"), u_rd=("rds","sum"),
        u_wl=("wls","sum"), u_wf=("wf","sum"), u_wr=("wr","sum"),
        u_last=("ld","min"), u_avgr=("mn_r","mean"),
        u_maxr=("mx_r","max"), u_span=("fd","max"))
    for d in [7, 14, 30]:
        t = ow[ow["event_ts"] >= ref - pd.Timedelta(days=d)]
        if t.empty: t2 = pd.DataFrame(columns=["user_id"])
        else: t2 = t.groupby("user_id", as_index=False).agg(**{
            f"u_ev{d}":("event_ts","size"), f"u_it{d}":("edition_id","nunique"), f"u_w{d}":("wf","sum")})
        uf = uf.merge(t2, on="user_id", how="left")
    uf = uf.merge(users_df[["user_id","gender","age","age_bkt"]], on="user_id", how="left")
    for c in uf.columns:
        if c != "user_id": uf[c] = uf[c].fillna(0 if c not in ("age","gender","age_bkt") else -1)
    uf["u_rdsh"] = (uf["u_rd"]/np.maximum(uf["u_ev"],1)).astype("float32")
    uf["u_rr"] = (uf["u_wr"]/np.maximum(uf["u_wf"],1e-6)).astype("float32")
    uf["u_tr"] = (uf.get("u_w14",0).fillna(0)/np.maximum(uf.get("u_w30",1).fillna(1),1e-6)).astype("float32")
    uf["u_int"] = (uf["u_ev"]/np.maximum(uf["u_span"],1)).astype("float32")  # activity intensity
    # -- Item features --
    itf = pair.groupby("edition_id", as_index=False).agg(
        i_us=("user_id","nunique"), i_ev=("n","sum"), i_rd=("rds","sum"),
        i_wl=("wls","sum"), i_wf=("wf","sum"), i_wr=("wr","sum"),
        i_last=("ld","min"), i_avgr=("mn_r","mean"),
        i_maxr=("mx_r","max"), i_avgld=("ld","mean"))
    for d in [7, 14, 30]:
        t = ow[ow["event_ts"] >= ref - pd.Timedelta(days=d)]
        if t.empty: t2 = pd.DataFrame(columns=["edition_id"])
        else: t2 = t.groupby("edition_id", as_index=False).agg(**{
            f"i_ev{d}":("event_ts","size"), f"i_us{d}":("user_id","nunique"), f"i_w{d}":("wf","sum")})
        itf = itf.merge(t2, on="edition_id", how="left")
    itf = itf.merge(ed_meta[["edition_id","book_id","author_id","pub_year","age_restr",
        "language_id","publisher_id","tlen","dlen","genre_cnt","age_proxy","tdlen"]],
        on="edition_id", how="left")
    for c in itf.columns:
        if c != "edition_id": itf[c] = itf[c].fillna(0)
    itf["i_rdsh"] = (itf["i_rd"]/np.maximum(itf["i_ev"],1)).astype("float32")
    itf["i_rr"] = (itf["i_wr"]/np.maximum(itf["i_wf"],1e-6)).astype("float32")
    itf["i_tr7"] = (itf.get("i_w7",0).fillna(0)/np.maximum(itf.get("i_w30",1).fillna(1),1e-6)).astype("float32")
    itf["i_tr14"] = (itf.get("i_w14",0).fillna(0)/np.maximum(itf.get("i_w30",1).fillna(1),1e-6)).astype("float32")
    itf["i_hot"] = (itf.get("i_w7",0).fillna(0)/np.maximum(itf["i_wf"],1e-6)).astype("float32")
    itf["i_prk"] = itf["i_wr"].rank(method="dense",ascending=False).astype("float32")
    itf["i_log_us"] = np.log1p(itf["i_us"]).astype("float32")
    # Author aggregate features
    au_stats = pair.merge(ed_meta[["edition_id","author_id"]], on="edition_id", how="left")
    au_stats = au_stats.groupby("author_id", as_index=False).agg(
        au_pop=("wr","sum"), au_items=("edition_id","nunique"), au_users=("user_id","nunique"))
    itf = itf.merge(au_stats, on="author_id", how="left")
    for c in ["au_pop","au_items","au_users"]: itf[c] = itf[c].fillna(0).astype("float32")
    # -- Affinity --
    pm = pair.merge(ed_meta[["edition_id","author_id","book_id","language_id","publisher_id"]], on="edition_id", how="left")
    def _agg(col, pre):
        o = pm.groupby(["user_id",col], as_index=False).agg(**{
            f"{pre}_w":("wf","sum"), f"{pre}_wr":("wr","sum"),
            f"{pre}_n":("edition_id","nunique"), f"{pre}_last":("ld","min")})
        o[f"{pre}_rk"] = o.groupby("user_id")[f"{pre}_wr"].rank(method="dense",ascending=False).astype("float32")
        return o
    ua=_agg("author_id","ua"); ub=_agg("book_id","ub"); ul=_agg("language_id","ul"); up=_agg("publisher_id","up")
    pg = pair[["user_id","edition_id","wf","wr","ld"]].merge(ed_gn, on="edition_id", how="left")
    pg = pg[pg["genre_id"].notna()]; pg["genre_id"] = pg["genre_id"].astype("int64")
    ug = pg.groupby(["user_id","genre_id"], as_index=False).agg(
        ug_w=("wf","sum"), ug_wr=("wr","sum"), ug_n=("edition_id","nunique"), ug_last=("ld","min"))
    ug["ug_rk"] = ug.groupby("user_id")["ug_wr"].rank(method="dense",ascending=False).astype("float32")
    # -- Seeds --
    rec = pair.sort_values(["user_id","ld","wr"], ascending=[True,True,False]).groupby("user_id").head(cfg.seed_n)
    rec["sw"] = (1.0/(1+rec["ld"]/7)*(1+0.15*rec["rds"])).astype("float32")
    hvy = pair.sort_values(["user_id","wr","wf"], ascending=[True,False,False]).groupby("user_id").head(cfg.seed_h)
    hvy["sw"] = np.log1p(hvy["wr"]+hvy["wf"]).astype("float32")
    seeds = pd.concat([rec[["user_id","edition_id","sw","ld","rds"]],
                        hvy[["user_id","edition_id","sw","ld","rds"]]], ignore_index=True)
    seeds = seeds.groupby(["user_id","edition_id"], as_index=False).agg(
        sw=("sw","max"), ld=("ld","min"), rds=("rds","max"))
    # -- Popularity --
    ipop = pair.groupby("edition_id", as_index=False).agg(pop=("wr","sum"), pop_u=("user_id","nunique"))
    ipop["pop_rk"] = ipop["pop"].rank(method="dense",ascending=False)
    gpop = ipop.sort_values("pop",ascending=False).head(cfg.top_glob)
    aipop = pm.groupby(["author_id","edition_id"], as_index=False).agg(ai_sc=("wr","sum"))
    aipop["ai_rk"] = aipop.groupby("author_id")["ai_sc"].rank(method="dense",ascending=False).astype("float32")
    aipop = aipop[aipop["ai_rk"]<=cfg.top_ent]
    bipop = pm.groupby(["book_id","edition_id"], as_index=False).agg(bi_sc=("wr","sum"))
    bipop["bi_rk"] = bipop.groupby("book_id")["bi_sc"].rank(method="dense",ascending=False).astype("float32")
    bipop = bipop[bipop["bi_rk"]<=15]
    gipop = pg.groupby(["genre_id","edition_id"], as_index=False).agg(gi_sc=("wr","sum"))
    gipop["gi_rk"] = gipop.groupby("genre_id")["gi_sc"].rank(method="dense",ascending=False).astype("float32")
    gipop = gipop[gipop["gi_rk"]<=cfg.top_ent]
    ou = pair[["user_id"]].drop_duplicates().merge(users_df[["user_id","gender","age_bkt"]], on="user_id", how="left")
    uir = pair.merge(ou, on="user_id", how="left")
    dpop = uir.groupby(["gender","age_bkt","edition_id"], as_index=False).agg(d_sc=("wr","sum"))
    dpop["d_rk"] = dpop.groupby(["gender","age_bkt"])["d_sc"].rank(method="dense",ascending=False).astype("float32")
    dpop = dpop[dpop["d_rk"]<=cfg.top_demo]
    genp = uir.groupby(["gender","edition_id"], as_index=False).agg(g_sc=("wr","sum"))
    genp["g_rk"] = genp.groupby("gender")["g_sc"].rank(method="dense",ascending=False).astype("float32")
    genp = genp[genp["g_rk"]<=cfg.top_demo]
    del ow, uir, ou; gc.collect()
    # ALS
    als1 = ALS(cfg.als1_f, cfg.als1_reg, cfg.als1_it, cfg.seed).fit(pair, "wf"); gc.collect()
    als2 = ALS(cfg.als2_f, cfg.als2_reg, cfg.als2_it, cfg.seed+7).fit(pair, "wr"); gc.collect()
    knn = fit_knn(pair, als1)
    seen = pair[["user_id","edition_id"]].drop_duplicates()
    return dict(ref=ref, pair=pair, uf=uf, itf=itf, ua=ua, ub=ub, ul=ul, up=up, ug=ug, pm=pm,
        seeds=seeds, gpop=gpop, aipop=aipop, bipop=bipop, gipop=gipop, dpop=dpop, genp=genp,
        ua_top=ua[ua["ua_rk"]<=cfg.top_au], ub_top=ub[ub["ub_rk"]<=cfg.top_bk],
        ug_top=ug[ug["ug_rk"]<=cfg.top_gn], als={"a1":als1,"a2":als2}, knn=knn, seen=seen)

# ══════════════════════════════════════════════════
# CANDIDATES
# ══════════════════════════════════════════════════
def gen_cands(ctx, uids):
    S = {}
    S["a1"] = ctx["als"]["a1"].recommend(uids, cfg.als1_topk).rename(columns={"rank":"a1_rk","score":"a1_sc"})
    S["a2"] = ctx["als"]["a2"].recommend(uids, cfg.als2_topk).rename(columns={"rank":"a2_rk","score":"a2_sc"})
    gc.collect()
    seeds = ctx["seeds"][ctx["seeds"]["user_id"].isin(uids)]
    # KNN
    for mode, topk, pre in [("lat",cfg.knn_lat,"lk"),("sp",cfg.knn_sp,"sk")]:
        nbr = q_nbrs(ctx["als"]["a1"], seeds["edition_id"].values, ctx["knn"]["lat_nn"], topk) if mode=="lat" else q_sp_nbrs(seeds["edition_id"].values, ctx["knn"], topk)
        if nbr.empty: S[pre]=pd.DataFrame(columns=["user_id","edition_id"]); continue
        mg = seeds[["user_id","edition_id","sw"]].rename(columns={"edition_id":"sid"}).merge(nbr, on="sid")
        mg[f"{pre}_ws"] = (mg["sw"]*mg["sim"]).astype("float32")
        S[pre] = mg.groupby(["user_id","edition_id"], as_index=False).agg(**{
            f"{pre}_sc":(f"{pre}_ws","sum"), f"{pre}_ms":("sim","max"),
            f"{pre}_cnt":("sim","size"), f"{pre}_rk":("nr","min")})
    # Co-occurrence
    S["cooc"] = build_cooc(ctx["pair"], ctx["seeds"], uids); gc.collect()
    # Text retrieval
    S["txt"] = txt_cands(ctx["seeds"], uids, cfg.txt_topk)
    # Entity pop
    for etop,eipop,ecol,ewcol,escol,ercol,pre in [
        (ctx["ua_top"],ctx["aipop"],"author_id","ua_wr","ai_sc","ai_rk","au"),
        (ctx["ub_top"],ctx["bipop"],"book_id","ub_wr","bi_sc","bi_rk","bk"),
        (ctx["ug_top"],ctx["gipop"],"genre_id","ug_wr","gi_sc","gi_rk","gn")]:
        ut=etop[etop["user_id"].isin(uids)]
        if ut.empty: S[pre]=pd.DataFrame(columns=["user_id","edition_id"]); continue
        mg=ut.merge(eipop,on=ecol,how="inner"); mg[f"{pre}_sc"]=(mg[ewcol]*mg[escol]).astype("float32")
        S[pre]=mg.groupby(["user_id","edition_id"],as_index=False).agg(**{f"{pre}_sc":(f"{pre}_sc","max"),f"{pre}_rk":(ercol,"min")})
    # Demo/glob
    ud=users_df[users_df["user_id"].isin(uids)][["user_id","gender","age_bkt"]]
    dm=ud.merge(ctx["dpop"],on=["gender","age_bkt"],how="left").dropna(subset=["edition_id"])
    gn=ud.merge(ctx["genp"],on="gender",how="left").dropna(subset=["edition_id"])
    if not dm.empty: dm["edition_id"]=dm["edition_id"].astype("int64")
    if not gn.empty: gn["edition_id"]=gn["edition_id"].astype("int64")
    S["demo"]=dm[["user_id","edition_id","d_sc","d_rk"]] if not dm.empty else pd.DataFrame(columns=["user_id","edition_id"])
    S["gend"]=gn[["user_id","edition_id","g_sc","g_rk"]] if not gn.empty else pd.DataFrame(columns=["user_id","edition_id"])
    gp=ctx["gpop"][["edition_id","pop","pop_rk"]].copy()
    if not gp.empty and len(uids)>0:
        udf=pd.DataFrame({"user_id":uids.astype(np.int64),"_t":1}); gp["_t"]=1
        S["glob"]=udf.merge(gp,on="_t").drop(columns="_t")
    else: S["glob"]=pd.DataFrame(columns=["user_id","edition_id"])
    # Merge
    parts=[df[["user_id","edition_id"]] for df in S.values() if not df.empty]
    if not parts: return pd.DataFrame(columns=["user_id","edition_id","pre_score"])
    pool=pd.concat(parts,ignore_index=True).drop_duplicates()
    for df in S.values():
        if df.empty: continue
        extra=[c for c in df.columns if c not in ("user_id","edition_id")]
        if extra: pool=pool.merge(df[["user_id","edition_id"]+extra],on=["user_id","edition_id"],how="left")
    pool=pool.merge(ctx["seen"].assign(_s=1),on=["user_id","edition_id"],how="left")
    pool=pool[pool["_s"].isna()].drop(columns="_s")
    # Pre-score
    sc=np.zeros(len(pool),dtype=np.float32)
    for col,w in [("a1_rk",2.8),("a2_rk",2.2),("lk_sc",1.6),("sk_sc",1.3),
                    ("au_rk",1.1),("bk_rk",0.9),("gn_rk",0.8),("d_rk",0.55),("g_rk",0.35),("pop_rk",0.45)]:
        if col not in pool.columns: continue
        if "_sc" in col: sc+=w*pool[col].fillna(0).values.astype(np.float32)
        else: sc+=w*r2s(pool[col].fillna(1e9)).values.astype(np.float32)
    if "cooc_sc" in pool.columns:
        mx=pool["cooc_sc"].max(); 
        if mx and mx>0: sc+=1.5*(pool["cooc_sc"].fillna(0)/mx).values.astype(np.float32)
    if "txt_sim" in pool.columns:
        sc+=1.2*pool["txt_sim"].fillna(0).values.astype(np.float32)
    rcols=[c for c in pool.columns if c.endswith("_rk")]
    pool["src_cnt"]=pool[rcols].notna().sum(axis=1).astype("int16") if rcols else 0
    sc+=0.15*pool["src_cnt"].values.astype(np.float32)
    pool["pre_score"]=sc
    pool=pool.sort_values(["user_id","pre_score"],ascending=[True,False]).groupby("user_id").head(cfg.max_cands).reset_index(drop=True)
    del S; gc.collect()
    return pool

# ══════════════════════════════════════════════════
# FEATURES
# ══════════════════════════════════════════════════
def mk_features(ctx, pool, hidden=None):
    df = pool.copy()
    df = df.merge(ctx["itf"], on="edition_id", how="left")
    df = df.merge(ctx["uf"], on="user_id", how="left")
    for aff,cols,keys in [
        (ctx["ua"],["ua_w","ua_wr","ua_n","ua_last","ua_rk"],["user_id","author_id"]),
        (ctx["ub"],["ub_w","ub_wr","ub_n","ub_last","ub_rk"],["user_id","book_id"]),
        (ctx["ul"],["ul_w","ul_wr","ul_n","ul_last","ul_rk"],["user_id","language_id"]),
        (ctx["up"],["up_w","up_wr","up_n","up_last","up_rk"],["user_id","publisher_id"])]:
        if all(k in df.columns for k in keys): df=df.merge(aff[keys+cols],on=keys,how="left")
    # ALS pair
    for key,pre in [("a1","a1"),("a2","a2")]:
        sc,co=ctx["als"][key].score_pairs(df["user_id"].values,df["edition_id"].values)
        df[f"{pre}_ps"]=sc; df[f"{pre}_pc"]=co
        # ALS user/item norms
        als_m = ctx["als"][key]
        un = np.zeros(len(df), dtype=np.float32); ine = np.zeros(len(df), dtype=np.float32)
        mask = np.array([int(u) in als_m.u2i and int(i) in als_m.i2i for u,i in zip(df["user_id"].values, df["edition_id"].values)])
        if mask.any():
            ui=np.array([als_m.u2i[int(u)] for u in df["user_id"].values[mask]])
            ii=np.array([als_m.i2i[int(i)] for i in df["edition_id"].values[mask]])
            un[mask]=np.linalg.norm(als_m.uf[ui],axis=1); ine[mask]=np.linalg.norm(als_m.itf[ii],axis=1)
        df[f"{pre}_un"]=un; df[f"{pre}_in"]=ine; df[f"{pre}_ng"]=np.abs(un-ine).astype("float32")
    # Text similarity between user profile and item
    try:
        user_seed_map = ctx["seeds"].groupby("user_id").apply(
            lambda g: list(zip(g["edition_id"].astype(int).tolist(), g["sw"].tolist()))).to_dict()
        txt_sims = np.zeros(len(df), dtype=np.float32)
        for i in range(len(df)):
            uid, eid = int(df.iloc[i]["user_id"]), int(df.iloc[i]["edition_id"])
            if eid not in eid_to_txtidx: continue
            seeds_list = user_seed_map.get(uid, [])
            if not seeds_list: continue
            sidxs = [eid_to_txtidx[s] for s,_ in seeds_list if s in eid_to_txtidx]
            if not sidxs: continue
            sws = np.array([w for s,w in seeds_list if s in eid_to_txtidx], dtype=np.float32)
            sws /= sws.sum()
            uvec = (txt_emb_n[sidxs] * sws[:,None]).sum(axis=0)
            uvec /= max(np.linalg.norm(uvec), 1e-12)
            txt_sims[i] = float(uvec @ txt_emb_n[eid_to_txtidx[eid]])
        df["txt_pair_sim"] = txt_sims
    except Exception as e:
        print(f"  txt_pair_sim skip: {e}")
        df["txt_pair_sim"] = 0.0
    # Genre match
    try:
        utg=ctx["ug"].sort_values(["user_id","ug_wr"],ascending=[True,False]).groupby("user_id").head(5)[["user_id","genre_id"]]
        dg=df[["user_id","edition_id"]].merge(ed_gn[["edition_id","genre_id"]],on="edition_id",how="left"); dg=dg[dg["genre_id"].notna()]
        dg=dg.merge(utg.assign(_m=1),on=["user_id","genre_id"],how="left")
        gm=dg.groupby(["user_id","edition_id"],as_index=False).agg(gm_cnt=("_m","sum"),gm_tot=("genre_id","size"))
        gm["gm_ratio"]=(gm["gm_cnt"].fillna(0)/np.maximum(gm["gm_tot"],1)).astype("float32")
        df=df.merge(gm[["user_id","edition_id","gm_cnt","gm_ratio"]],on=["user_id","edition_id"],how="left")
    except: pass
    for c in ["gm_cnt","gm_ratio"]: df[c]=df.get(c,pd.Series(0,index=df.index)).fillna(0).astype("float32")
    # Cross features
    df["x_hot"]=(df["u_rr"].fillna(0)*df["i_hot"].fillna(0)).astype("float32")
    df["x_rd"]=(df["u_rdsh"].fillna(0)*df["i_rdsh"].fillna(0)).astype("float32")
    df["x_tr"]=(df["u_tr"].fillna(0)*df["i_tr14"].fillna(0)).astype("float32")
    df["als_diff"]=(df["a2_ps"].fillna(0)-df["a1_ps"].fillna(0)).astype("float32")
    df["als_cdiff"]=(df["a2_pc"].fillna(0)-df["a1_pc"].fillna(0)).astype("float32")
    df["i_niche"]=(1.0/np.maximum(df["i_us"].fillna(1),1)).astype("float32")
    df["i_pop_ua"]=(df["i_ev"].fillna(0)/np.maximum(df["u_ev"].fillna(1),1)).astype("float32")
    df["age_gap"]=(df["age"].fillna(-1)-df["age_restr"].fillna(-1)).astype("float32")
    df["pub_gap"]=(2025-df["pub_year"].fillna(2025)).astype("float32")
    df["u_x_aupop"]=(df["ua_wr"].fillna(0)/np.maximum(df["au_pop"].fillna(1),1e-6)).astype("float32")
    df["log_ld"]=np.log1p(df["i_last"].fillna(999)).astype("float32")
    # Fill
    num=df.select_dtypes(include=[np.number,"bool"]).columns
    for c in num: df[c]=df[c].fillna(0)
    cat_cols=["author_id","book_id","language_id","publisher_id","gender","age_bkt"]
    for c in cat_cols:
        if c in df.columns: df[c]=df[c].fillna(-1).astype(str)
    if hidden is not None:
        df=df.merge(hidden.assign(label=1),on=["user_id","edition_id"],how="left")
        df["label"]=df["label"].fillna(0).astype("int8")
    return df

# ══════════════════════════════════════════════════
# FOLDS
# ══════════════════════════════════════════════════
def make_folds():
    mx=inter["event_ts"].max().normalize()+pd.Timedelta(days=1)
    mn=inter["event_ts"].min().normalize()
    folds=[]
    for i in range(cfg.n_folds):
        end=mx-pd.Timedelta(days=cfg.fold_days*(cfg.n_folds-1-i))
        start=end-pd.Timedelta(days=cfg.fold_days)
        if start<=mn: continue
        folds.append((f"f{i}",start,end,cfg.seed+100+i))
    return folds

def build_masked(start,end,seed):
    before=inter[inter["event_ts"]<start][["user_id","edition_id"]].drop_duplicates()
    inc=inter[(inter["event_ts"]>=start)&(inter["event_ts"]<end)].copy()
    elig=inc.merge(before.assign(_s=1),on=["user_id","edition_id"],how="left")
    elig=elig[elig["_s"].isna()].drop(columns="_s")
    if elig.empty:
        return inter[inter["event_ts"]<end].copy(),pd.DataFrame(columns=["user_id","edition_id"]),np.array([],dtype=np.int64)
    ep=elig.groupby(["user_id","edition_id"],as_index=False).agg(
        cnt=("event_ts","size"),hr=("event_type",lambda x:int((x==2).any())))
    rng=np.random.default_rng(seed); hi=[]
    for uid,grp in ep.groupby("user_id"):
        n=len(grp); nr=float(np.clip(cfg.hide_rate*rng.lognormal(0,0.35),0.05,0.6))
        nh=max(0,min(int(round(n*nr)),n))
        if nh==0 and n>=3 and rng.random()<0.35: nh=1
        if nh==0: continue
        p=(1+0.5*grp["hr"].values+0.25*np.log1p(grp["cnt"].values)).astype(np.float64); p/=p.sum()
        hi.extend(rng.choice(grp.index.values,size=nh,replace=False,p=p).tolist())
    hidden=ep.loc[hi,["user_id","edition_id"]].drop_duplicates().reset_index(drop=True)
    obs=inter[inter["event_ts"]<end].copy()
    if not hidden.empty:
        obs=obs.merge(hidden.assign(_h=1),on=["user_id","edition_id"],how="left")
        obs=obs[obs["_h"].isna()].drop(columns="_h")
    return obs.reset_index(drop=True),hidden,np.array(sorted(hidden["user_id"].unique()),dtype=np.int64)

def prep_fold(name,start,end,seed):
    print(f"\n[{name}] {'='*50}")
    obs,hidden,fusers=build_masked(start,end,seed)
    ctx=build_ctx(obs,end); del obs; gc.collect()
    pool=gen_cands(ctx,fusers)
    miss=hidden.merge(pool[["user_id","edition_id"]],on=["user_id","edition_id"],how="left",indicator=True)
    miss=miss[miss["_merge"]=="left_only"][["user_id","edition_id"]]; miss["pre_score"]=0.0
    if not miss.empty:
        pool=pd.concat([pool,miss],ignore_index=True,sort=False).drop_duplicates(["user_id","edition_id"])
        print(f"  +{len(miss)} missed pos")
    feat=mk_features(ctx,pool,hidden)
    base=ev_scored(feat,hidden,"pre_score")
    print(f"  NDCG@20={base:.5f} rows={len(feat)} pos={int(feat['label'].sum())}")
    del ctx,pool; gc.collect()
    return {"hidden":hidden,"feat":feat}

folds=make_folds()
for f in folds: print(f"  {f[0]}: {f[1]} -> {f[2]}")
fold_arts=[]
for name,start,end,seed in folds:
    art=prep_fold(name,start,end,seed)
    if art["feat"].empty or art["hidden"].empty: print(f"Skip {name}"); continue
    fold_arts.append(art); gc.collect()

# ══════════════════════════════════════════════════
# CATBOOST
# ══════════════════════════════════════════════════
from catboost import CatBoostRanker, Pool as CBP

def ds_neg(df,mx,seed):
    if "label" not in df.columns: return df
    rng=np.random.default_rng(seed); pos=df[df["label"]==1]; neg=df[df["label"]==0]
    if neg.empty: return pos
    neg=neg.sort_values(["user_id","pre_score"],ascending=[True,False])
    hard=neg.groupby("user_id").head(mx//2)
    rest=neg.merge(hard[["user_id","edition_id"]].assign(_h=1),on=["user_id","edition_id"],how="left")
    rest=rest[rest["_h"].isna()].drop(columns="_h"); sampled=[]
    nr=mx-mx//2
    for _,grp in rest.groupby("user_id"):
        if len(grp)<=nr: sampled.append(grp)
        else: sampled.append(grp.loc[rng.choice(grp.index.values,nr,replace=False)])
    return pd.concat([pos,hard]+sampled,ignore_index=True).sort_values(["user_id","pre_score"],ascending=[True,False]).reset_index(drop=True)

assert len(fold_arts)>=2
val_art=fold_arts[-1]; train_arts=fold_arts[:-1]
train_df=pd.concat([a["feat"] for a in train_arts],ignore_index=True); valid_df=val_art["feat"].copy()
for a in train_arts: del a["feat"]
gc.collect()
train_df=ds_neg(train_df,cfg.max_neg,cfg.seed)
gs=train_df.groupby("user_id")["edition_id"].size(); gp=train_df.groupby("user_id")["label"].sum()
good=set(gs[gs>=cfg.min_grp].index)&set(gp[gp>0].index); train_df=train_df[train_df["user_id"].isin(good)].copy()
DROP={"label"}; KEYS=["user_id","edition_id"]; CAT=["author_id","book_id","language_id","publisher_id","gender","age_bkt"]
feat_cols=[c for c in train_df.columns if c not in DROP and c not in KEYS]
cat_cols=[c for c in CAT if c in feat_cols]; num_cols=[c for c in feat_cols if c not in cat_cols]
for c in num_cols: train_df[c]=pd.to_numeric(train_df[c],errors="coerce").fillna(0).astype("float32"); valid_df[c]=pd.to_numeric(valid_df[c],errors="coerce").fillna(0).astype("float32")
for c in cat_cols: train_df[c]=train_df[c].fillna("UNK").astype(str); valid_df[c]=valid_df[c].fillna("UNK").astype(str)
print(f"\nTrain:{train_df.shape} pos={int(train_df['label'].sum())}  Valid:{valid_df.shape} pos={int(valid_df['label'].sum())}")
print(f"Features: {len(feat_cols)} ({len(num_cols)} num + {len(cat_cols)} cat)")

tp=CBP(train_df[feat_cols],train_df["label"],group_id=train_df["user_id"],cat_features=cat_cols)
vp=CBP(valid_df[feat_cols],valid_df["label"],group_id=valid_df["user_id"],cat_features=cat_cols)
pa=dict(loss_function="YetiRank",eval_metric="NDCG:top=20",task_type="CPU",random_seed=cfg.seed,iterations=cfg.cb1_it,learning_rate=cfg.cb1_lr,depth=10,l2_leaf_reg=8,bootstrap_type="Bernoulli",subsample=0.8,verbose=200,od_type="Iter",od_wait=200,border_count=254)
pb=dict(loss_function="QuerySoftMax",eval_metric="NDCG:top=20",task_type="CPU",random_seed=cfg.seed+17,iterations=cfg.cb2_it,learning_rate=cfg.cb2_lr,depth=10,l2_leaf_reg=6,bootstrap_type="Bernoulli",subsample=0.85,verbose=200,od_type="Iter",od_wait=200,border_count=254)
print("Training A (YetiRank)..."); ma=CatBoostRanker(**pa); ma.fit(tp,eval_set=vp,use_best_model=True)
print("Training B (QuerySoftMax)..."); mb=CatBoostRanker(**pb); mb.fit(tp,eval_set=vp,use_best_model=True)
va,vb=ma.predict(vp),mb.predict(vp)
ve=valid_df[["user_id","edition_id"]].copy(); ve["pa"],ve["pb"],ve["pre"]=va,vb,valid_df["pre_score"].values
sa=ev_scored(ve.rename(columns={"pa":"score"}),val_art["hidden"],"score")
sb=ev_scored(ve.rename(columns={"pb":"score"}),val_art["hidden"],"score")
print(f"\nVal: YetiRank={sa:.6f} QuerySoftMax={sb:.6f}")
bw,bs=0.5,-1
for w in np.linspace(0,1,21):
    t=ve[["user_id","edition_id"]].copy(); t["score"]=(w*ve["pa"]+(1-w)*ve["pb"]).astype(np.float32)
    s=ev_scored(t,val_art["hidden"],"score")
    if s>bs: bs,bw=s,float(w)
ba,bsc=0.0,bs
for alpha in np.linspace(0,0.25,11):
    t=ve[["user_id","edition_id"]].copy(); ms=bw*ve["pa"]+(1-bw)*ve["pb"]
    pn=ve["pre"]/max(ve["pre"].std(),1e-6)*ms.std()
    t["score"]=((1-alpha)*ms+alpha*pn).astype(np.float32)
    s=ev_scored(t,val_art["hidden"],"score")
    if s>bsc: bsc,ba=s,float(alpha)
print(f"Blend: w_a={bw:.2f} alpha={ba:.3f} NDCG={bsc:.6f}")

# RETRAIN
del fold_arts; gc.collect()
all_df=pd.concat([train_df,valid_df],ignore_index=True); del train_df,valid_df,tp,vp; gc.collect()
all_df=ds_neg(all_df,cfg.max_neg,cfg.seed+123)
gs=all_df.groupby("user_id")["edition_id"].size(); gp=all_df.groupby("user_id")["label"].sum()
good=set(gs[gs>=cfg.min_grp].index)&set(gp[gp>0].index); all_df=all_df[all_df["user_id"].isin(good)].copy()
for c in num_cols: all_df[c]=pd.to_numeric(all_df[c],errors="coerce").fillna(0).astype("float32")
for c in cat_cols: all_df[c]=all_df[c].fillna("UNK").astype(str)
fp=CBP(all_df[feat_cols],all_df["label"],group_id=all_df["user_id"],cat_features=cat_cols)
bia=ma.get_best_iteration(); bib=mb.get_best_iteration()
bia=pa["iterations"] if bia is None or bia<=0 else int(bia*1.08)
bib=pb["iterations"] if bib is None or bib<=0 else int(bib*1.08)
print(f"\nRetrain A({bia})..."); fma=CatBoostRanker(**{**pa,"iterations":bia}); fma.fit(fp,verbose=200)
print(f"Retrain B({bib})..."); fmb=CatBoostRanker(**{**pb,"iterations":bib}); fmb.fit(fp,verbose=200)
del all_df,fp,ma,mb; gc.collect()

# ══════════════════════════════════════════════════
# TEST
# ══════════════════════════════════════════════════
print("\nTest inference...")
full_ref=inter["event_ts"].max().normalize()+pd.Timedelta(days=1)
full_ctx=build_ctx(inter,full_ref)
tuids=targets["user_id"].drop_duplicates().sort_values().values.astype(np.int64)
print(f"Candidates for {len(tuids)} users...")
tp2=gen_cands(full_ctx,tuids); print(f"  pool: {tp2.shape}")
tf2=mk_features(full_ctx,tp2,hidden=None)
for c in num_cols:
    if c in tf2.columns: tf2[c]=pd.to_numeric(tf2[c],errors="coerce").fillna(0).astype("float32")
    else: tf2[c]=0.0
for c in cat_cols:
    if c in tf2.columns: tf2[c]=tf2[c].fillna("UNK").astype(str)
    else: tf2[c]="UNK"
for c in feat_cols:
    if c not in tf2.columns: tf2[c]=0.0 if c in num_cols else "UNK"
tpool=CBP(tf2[feat_cols],cat_features=cat_cols)
tpa,tpb=fma.predict(tpool),fmb.predict(tpool)
scored=tf2[["user_id","edition_id","pre_score"]].copy()
scored["pa"],scored["pb"]=tpa.astype(np.float32),tpb.astype(np.float32)
ms=bw*scored["pa"]+(1-bw)*scored["pb"]
if ba>0:
    pn=scored["pre_score"]/max(scored["pre_score"].std(),1e-6)*ms.std()
    scored["final_score"]=((1-ba)*ms+ba*pn).astype(np.float32)
else: scored["final_score"]=ms.astype(np.float32)
del tf2,tpool,tp2; gc.collect()

# ══════════════════════════════════════════════════
# SUBMISSION
# ══════════════════════════════════════════════════
print("Building submission...")
ia=ed_meta.set_index("edition_id")["author_id"].to_dict()
sm=defaultdict(set)
for r in full_ctx["seen"].itertuples(index=False): sm[int(r.user_id)].add(int(r.edition_id))
ud=users_df[["user_id","gender","age_bkt"]]; dm_m,gn_m=defaultdict(list),defaultdict(list)
for r in full_ctx["dpop"].sort_values(["gender","age_bkt","d_rk"]).itertuples(index=False): dm_m[(int(r.gender),int(r.age_bkt))].append(int(r.edition_id))
for r in full_ctx["genp"].sort_values(["gender","g_rk"]).itertuples(index=False): gn_m[int(r.gender)].append(int(r.edition_id))
gl=full_ctx["gpop"].sort_values("pop_rk")["edition_id"].astype(int).tolist()
fb={}
for r in ud[ud["user_id"].isin(tuids)].itertuples(index=False):
    seq=dm_m.get((int(r.gender),int(r.age_bkt)),[])+gn_m.get(int(r.gender),[])+gl
    u=[]; s=set()
    for x in seq:
        if x not in s: u.append(x); s.add(x)
    fb[int(r.user_id)]=u
for uid in tuids:
    if int(uid) not in fb: fb[int(uid)]=gl[:]
scored=scored.sort_values(["user_id","final_score"],ascending=[True,False])
grouped={}
for uid,g in scored.groupby("user_id"): grouped[int(uid)]=list(zip(g["edition_id"].astype(int).tolist(),g["final_score"].tolist()))
sub=[]; all_ed=ed_meta["edition_id"].astype(int).tolist()
for uid in tqdm(tuids.astype(int),desc="Sub"):
    recs,used=[],set(); seen=sm.get(uid,set())
    cands=grouped.get(uid,[]); rem=[(e,s) for e,s in cands if e not in seen][:TOP_K*5]
    sel_a=[]
    for _ in range(min(TOP_K,len(rem))):
        if not rem: break
        bj,ba2=0,-1e18
        for j,(e,s) in enumerate(rem):
            a=ia.get(e,-1); pen=sum(0.12 for sa in sel_a if sa==a and a!=-1)
            if s-pen>ba2: ba2,bj=s-pen,j
        e,s=rem.pop(bj); recs.append(e); used.add(e); sel_a.append(ia.get(e,-1))
    if len(recs)<TOP_K:
        for e in fb.get(uid,[]):
            if e not in used and e not in seen: recs.append(e); used.add(e)
            if len(recs)==TOP_K: break
    if len(recs)<TOP_K:
        for e in all_ed:
            if e not in used and e not in seen: recs.append(e); used.add(e)
            if len(recs)==TOP_K: break
    for rank,e in enumerate(recs[:TOP_K],1): sub.append({"user_id":uid,"edition_id":e,"rank":rank})
submission=pd.DataFrame(sub)
assert len(submission)==len(tuids)*TOP_K
submission.to_csv(cfg.output_dir/"submission.csv",index=False)
print(f"Saved: {submission.shape}")
print(submission.head(20))
print("DONE!")

Data: /kaggle/input/datasets/artemnazemtsev/nto-ai
Loading data...
Building text embeddings...
  Text embeddings: (460599, 48), explained var: 0.066
  inter: (443278, 5)
  targets: (3862, 1)
  editions: (460599, 9)
implicit: True
  f0: 2025-09-02 00:00:00 -> 2025-10-02 00:00:00
  f1: 2025-10-02 00:00:00 -> 2025-11-01 00:00:00
  f2: 2025-11-01 00:00:00 -> 2025-12-01 00:00:00

[f0] ==================================================


  0%|          | 0/15 [00:00<?, ?it/s]

  ALS: u=(16598, 160) i=(101079, 160)


  0%|          | 0/12 [00:00<?, ?it/s]

  ALS: u=(16598, 96) i=(101079, 96)
  +10626 missed pos
